# 1.1 · PD training

*1. Experiment 1 · notebook 1.1 of the story.* ← [0.3 · the LGD prior](<../0. General/0.3_prior_visualisation_lgd.ipynb>) · [1.2 · LGD training](<1.2_lgd_training.ipynb>) →

**What every Experiment 1 arm did while it trained** — whether the run is
trustworthy, what the monitoring says about the levers, where it holds, and every arm in detail. It
reads the per-arm `output/manifests/exp1_pd__*` files, so it renders on a partial sweep and
re-runs cleanly as arms finish.

**Experiment 1 asks which prior.** A nano-scale TabICL-architecture model is trained **from scratch** on each prior for 12,500 steps of 64 datasets, sweeping `credit_fraction` (0 = the control, exactly TabICL's own prior; 0.5; 1) × `filter.mode` (`tabicl`, `banded`, `off`) × prior intensity (mild, aggressive) × 3 seeds — 15 priors, 45 arms. Every arm sees the same optimiser steps: **matched compute is the whole basis of the comparison** (`papers/2026/02_Qu_TabICLv2` §4.1).

**What "monitoring" means here — and what it may not be used for.** Every
`progress.every_datasets` datasets the model is scored on the **development split** of the real
credit datasets. That is monitoring, not the result: the result is the benchmark in
[1.3 · PD results](<1.3_pd_results.ipynb>), where the prior is *selected* on development data and *reported* on
the untouched holdout (`docs/EXPERIMENTAL_DESIGN.md` §5 — choosing on the holdout would invalidate
the experiment). Two rules decide what every curve below averages over:

- **Finished arms only.** An average over every arm changes composition wherever an unfinished arm
  stops, and the curve jumps for no reason but bookkeeping. Unfinished arms stay visible — dotted,
  hatched in the sweep map, and in their own panel.
- **Development datasets only, the same for every arm.** Curves from before the development-only
  monitoring protocol (23-09-2026) scored "the smallest few" datasets whatever their role, holdout
  ones included. Those holdout scores are *shown* (C1) but **never averaged**; an arm whose
  development monitor logged nothing is left out of an average and counted, never averaged over
  fewer datasets. Figure A2 draws exactly which cells count.

**How to read it.** Part A says whether the run can be trusted; Part B what development monitoring
says about each lever; Part C whether that holds on every dataset and off the credit domain; Part D
is every arm, for the reader who wants to check one.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder
# (notebooks/1. Experiment 1/), from notebooks/, or from the root — then work FROM the root,
# so relative paths (config/...) resolve exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import training_plots, figures, style, literature

style.apply()   # ONE shared style: identical colours in every figure of every notebook
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "pd"
EXP = "exp1"
# Reads output/manifests/ and output/results/ — whatever the run has written so far, so a PARTIAL
# sweep still renders. Constructing the saver clears THIS notebook's figure folder, and only it.
FIGS = figures.FigureSaver("1.1_pd_training")

## The colour key

One vocabulary for every figure in every notebook, so a reader who has understood one figure can
read the next without its legend. Blue is our credit prior, grey the unmodified TabICL prior (the
control), orange real data, teal a value from `tfm-library`, amber one from outside it. Wherever a
figure varies the **credit fraction**, it runs from the control's grey (cf = 0) to our prior's blue
(cf = 1), monotone in lightness so the three levels still separate in a greyscale print.

In [ ]:
FIGS.save(style.show_palette(), "palette",
    caption="The shared colour vocabulary used on every axis of this notebook: each swatch names the prior, data source, literature overlay or annotation it marks.");

## A · Is the run sound?

Before any number is read: which arms exist and finished, what each was scored on, what each cost, and whether every one of them actually learned.

### A1 · The sweep at a glance

Every arm in one grid — one row per configuration, grouped by credit fraction, one column per seed. The colour is the arm's final development ROC-AUC, so this map also previews Part B's answer; a hatched cell is still training and says how far it got. Read it for the *structure*: does a block of rows sit darker or lighter than the rest, and is one seed systematically high in every row — a shared-seed effect, which is exactly why every comparison below is read against the seed spread (B8).

In [ ]:
FIGS.save(training_plots.sweep_map(TASK, exp=EXP), "sweep_map",
    caption="Final development ROC-AUC of every PD arm as a heatmap, one row per configuration grouped by credit fraction and one column per seed; hatched cells are arms still training, annotated with the share trained.");

### A2 · What each arm was scored on

The monitoring map: one row per real dataset (development first, then holdout), one column per arm. **Only teal cells enter a development mean.** Grey cells are holdout datasets an arm happened to monitor under the old protocol — kept as evidence, never averaged; red cells are a monitor that logged only missing values. This is the figure that says how much of Part B rests on how many datasets and arms.

In [ ]:
FIGS.save(training_plots.monitoring_coverage(TASK, exp=EXP), "monitoring_coverage",
    caption="Monitoring coverage of every PD arm: one row per real dataset labelled development or holdout, one column per arm in sweep order, cells coloured by whether the dataset enters the development mean, is holdout, is missing or was not monitored.");

### A3 · What each arm cost

TabICL generates its prior on the CPU, so a filter that rejects most candidate tasks starves the GPU — invisible in the loss, obvious here. The predictability filter keeps a task only if a shallow ExtraTrees beats the mean on it (25 trees, a bootstrap p < 0.05 — `repositories/TabICL.txt` `should_filter`); `banded` additionally demands the signal fall in credit's weak band, so it rejects far more, and each of its arms costs several times the wall-clock of an `off` or `tabicl` arm.

In [ ]:
FIGS.save(training_plots.throughput(TASK, exp=EXP), "throughput",
    caption="Training speed per PD arm in steps per second (left) and the implied hours for the full run (right), grouped by filter mode; points are arms coloured by credit fraction and bars are group medians.");

### A4 · Did every arm learn?

The loss is the only quantity every arm optimises directly, so a reader checks it before any real-data curve: every arm should descend, none diverge. TabICLv2's ablations warn of a prior × architecture interaction that can make training diverge on the wrong prior (`papers/2026/02_Qu_TabICLv2` §4.4) — divergence would be a reportable outcome, never tuned away.

In [ ]:
FIGS.save(training_plots.training_loss(TASK, exp=EXP), "training_loss",
    caption="Training loss against optimisation step for every PD arm (grey), the mean of the finished arms (black), and the best and worst arms by final development ROC-AUC (highlighted).");

### A5 · Is every block learning?

One gradient-norm curve per architecture block — column encoder, row encoder, ICL blocks, head. Every stack should stay off the floor.

In [ ]:
FIGS.save(training_plots.gradient_flow(TASK, exp=EXP), "gradient_flow",
    caption="Mean per-block gradient L2 norm (column encoder, row encoder, ICL blocks, head) against training step on a logarithmic axis.");

### A6 · Gradient-to-weight ratio

The interpretable version of A5: a gradient of 0.01 is tiny against weights of 0.1 and enormous against 1e-5, so the ratio is what says whether a block is effectively frozen.

In [ ]:
FIGS.save(training_plots.weight_gradient_ratios(TASK, exp=EXP), "weight_gradient_ratios",
    caption="Mean per-block ratio of gradient norm to weight norm against training step on a logarithmic axis, one line per architecture block.");

## B · What does development monitoring say?

The development ROC-AUC over training and at its end, lever by lever. Development monitoring guides the eye; the benchmark in [1.3 · PD results](<1.3_pd_results.ipynb>) is what decides.

### B1 · Development ROC-AUC over training

The first sight of the model on real credit data it never trained on. A prior that helps should lift this curve earlier or higher than the control. Each arm is drawn in its credit-fraction colour, so a lever that dominates separates the bundle by colour.

In [ ]:
FIGS.save(training_plots.metric_over_training(TASK, exp=EXP), "metric_over_training",
    caption="Development ROC-AUC averaged over the development datasets every arm carries, against training step; one line per arm coloured by credit fraction, dotted while unfinished, with the mean of the finished arms in black.");

### B2 · Credit prior versus control

The experiment's comparison drawn as a curve: every credit-prior arm against the `credit_fraction = 0` control, which is by construction TabICL's own prior. The median and inter-quartile band keep the spread visible — no gap narrower than that band is read as an effect.

In [ ]:
FIGS.save(training_plots.credit_vs_control_over_training(TASK, exp=EXP), "credit_vs_control_over_training",
    caption="Development ROC-AUC against training step for credit-prior arms versus control arms: median lines with shaded inter-quartile bands across the finished arms in each group.");

### B3 · ROC-AUC by credit fraction

One lever isolated: every finished arm sharing a value of **credit fraction**, averaged. `credit_fraction` is the master switch — the share of every batch drawn from our prior. Mitra finds mixtures beat single priors (`papers/2025/10_Zhang_Mitra`), so an interior optimum is plausible.

In [ ]:
FIGS.save(training_plots.metric_by_lever(TASK, "credit_fraction", exp=EXP), "metric_by_lever_credit_fraction",
    caption="Development ROC-AUC against training step, one mean line per value of credit fraction, averaged over the finished arms that share each value.");

### B4 · ROC-AUC by filter mode

One lever isolated: every finished arm sharing a value of **filter mode**, averaged. `banded` removes rather than adds — it cannot be accused of adding capacity — and it goes against TabICLv2's own finding that filtering *improves* convergence (`papers/2026/02_Qu_TabICLv2`, Fig. 10).

In [ ]:
FIGS.save(training_plots.metric_by_lever(TASK, "filter", exp=EXP), "metric_by_lever_filter",
    caption="Development ROC-AUC against training step, one mean line per value of filter mode, averaged over the finished arms that share each value.");

### B5 · ROC-AUC by prior intensity

One lever isolated: every finished arm sharing a value of **prior intensity**, averaged. Intensity is the mild-versus-aggressive setting of the credit mechanisms: retail versus corporate default correlation for PD, light versus heavy boundary atoms for LGD. The control has no credit prior, so it has no intensity either.

In [ ]:
FIGS.save(training_plots.metric_by_lever(TASK, "intensity", exp=EXP), "metric_by_lever_intensity",
    caption="Development ROC-AUC against training step, one mean line per value of prior intensity, averaged over the finished arms that share each value.");

### B6 · Final score, lever by lever

The sweep read as a screen: every finished arm's final score, grouped by each lever in turn, each point coloured by its credit fraction. A lever whose groups differ only because they hold different fractions shows up as colour sorting inside the groups.

In [ ]:
FIGS.save(training_plots.final_metric_by_lever(TASK, exp=EXP), "final_metric_by_lever",
    caption="Final development ROC-AUC of every finished arm as points coloured by credit fraction, one panel per swept lever, with a horizontal bar at each group mean.");

### B7 · Each lever within each credit fraction

The sweep is **factorial**, so a lever pooled across credit fractions inherits the fraction's effect as spread — the pooled panels above cannot tell a real filter effect from a fraction effect in disguise. Here each fraction is its own series; a lever the control does not have (prior intensity) shows the control as a grey band to beat.

In [ ]:
FIGS.save(training_plots.lever_interaction(TASK, exp=EXP), "lever_interaction",
    caption="Final development ROC-AUC against each lever other than credit fraction, one series per credit fraction: points are finished arms, lines join the per-value means and whiskers show one standard deviation over seeds; the grey band is the control mean plus or minus one standard deviation.");

### B8 · Effect versus seed noise

Every configuration's seeds side by side, best on top. A difference between two configurations is only worth reading where it clearly exceeds the spread among one configuration's own seeds — the reason Experiment 1 runs three (`docs/EXPERIMENTAL_DESIGN.md`). Where only one seed per configuration carries a development score, the figure says so instead of inventing a spread.

In [ ]:
FIGS.save(training_plots.seed_spread(TASK, exp=EXP), "seed_spread",
    caption="Final development ROC-AUC of every configuration's seeds as points on one horizontal axis, configurations sorted by their mean with the range and mean marked per row; the dashed line is the control mean.");

## C · Where does it hold?

A mean can be carried by one easy dataset, and a credit gain can be bought with general ability. These figures check both.

### C1 · Every dataset

One panel per monitored dataset, credit against control. Development panels are what the means above are built from; **holdout panels are shown as the evidence they are and never enter a mean** — they belong to the benchmark. A panel title saying *missing in N arms* is a monitor that logged nothing for those arms.

In [ ]:
for _p in range(1, training_plots.per_dataset_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_dataset_curves(TASK, _p, exp=EXP), f"per_dataset_p{_p}",
        caption="Monitored ROC-AUC against training step, one panel per real dataset labelled with its role, credit arms (blue) against control arms (grey); holdout datasets are shown but never averaged.");

### C2 · Credit versus out-of-domain

Out-of-domain retention is a first-class axis (`docs/EXPERIMENTAL_DESIGN.md` §5.3): a prior that lifts credit by making the model worse everywhere else has not made it better. Credit and out-of-domain suites are each averaged over the datasets every finished arm carries in that domain.

In [ ]:
FIGS.save(training_plots.real_vs_ood(TASK, exp=EXP), "real_vs_ood",
    caption="Development ROC-AUC on the real-credit datasets (solid) and the out-of-domain suites (dashed) against training step, averaged separately over credit-prior and control arms.");

### C3 · Every development metric

Not only the headline: every metric the monitor records, so a prior that helps ranking but hurts calibration is caught — the axis the library calls a first-class selling point yet under-measured across adaptation regimes (`SYNTHESIS.md`).

In [ ]:
for _p in range(1, training_plots.metric_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.all_eval_metrics(TASK, _p, exp=EXP), f"eval_metrics_p{_p}",
        caption="Each logged development metric against training step, one panel per metric, credit-prior (blue) and control (grey) means over the finished arms; the arrow in each title marks the improving direction.");

## D · Every arm, in detail

For the reader who wants to check one arm rather than trust a mean.

### D1 · Every arm's curves

Each arm's own loss and development curve, in the sweep map's order, so an arm that behaved unlike its neighbours is easy to find. An unfinished arm's title says how far it got.

In [ ]:
for _p in range(1, training_plots.config_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_config(TASK, _p, exp=EXP), f"per_config_p{_p}",
        caption="Per-arm training curves against step in sweep-map order: train loss (grey) and development ROC-AUC (blue, right axis), one panel per arm, unfinished arms labelled with the share trained.");

### D2 · Best versus worst

The two extremes side by side — did the worst arm fail to descend, or descend to a worse place?

In [ ]:
FIGS.save(training_plots.best_and_worst(TASK, exp=EXP), "best_and_worst",
    caption="Train loss and development ROC-AUC against training step for the best and worst finished arm by final development ROC-AUC.");

### D3 · Hardware

Was the machine working? A data-starved run and a compute-bound one give the same loss curve but need opposite fixes (`docs/VSC.md`); utilisation tells them apart.

In [ ]:
FIGS.save(training_plots.hardware(TASK, exp=EXP), "hardware",
    caption="GPU utilisation, throughput and peak allocated memory against training step, pooled across arms; the dashed line marks 70 percent utilisation.");

## Summary

The sweep in text: the run (A), what development monitoring says (B), where it holds (C). Printed last, in the order of the sections above, so `output/All_Results.md` reads the
same story as this notebook — followed by the `tfm-library` sources it leans on (pin `e5ce016`) and
the figure inventory.

In [ ]:
print(training_plots.training_summary(TASK, exp=EXP))
print()
print(literature.references_md(["batch", "stage1_lr", "optimizer", "filter_rate_clf", "filter_pval", "filter_extratrees_n", "oprior_headline", "mitra", "calibration_gap", "merton_vasicek"]))
print()
print(FIGS.summary())